# GTSRB Research: Master 2x4 Factorial Benchmark (Test-Time BN-Adaptation)

This orchestrator evaluates the entire project ladder under **Test-Time Batch Normalization Adaptation (BN-Adapt)**:
$$\{\text{Baseline, AugMix, Frequency, CBM}\} \times \{\text{Static Inference, Shift-Gated BN-Adapt}\}$$

### Core Scientific Hypotheses Tested:
1. **Baseline Control:** What is the raw statistical benefit of prediction-time statistics realignment on unregularized ResNet-18 features?
2. **AugMix Peak Generalization:** Does correcting the noise statistic offset in AugMix unlock its superior Weather/Blur invariant features, pushing past 88.5% mCA?
3. **Frequency Interaction:** Are train-time Fourier spectral filtering and test-time batch statistics adaptation complementary or redundant?
4. **CBM Semantic Cushioning:** How does high-level concept bottlenecking interact with low-level statistic correction?

**Zero Training on Corruptions. Zero Labels. Zero Parameter Updates.**

In [ ]:
# --- Step 1: Environment Setup, Drive Mount & GPU Check ---
import os
import sys
from google.colab import drive
drive.mount('/content/drive')

# Ensure project folder exists
%cd /content/ML_Project

# Ensure ML_Project is in PYTHONPATH
if '/content/ML_Project' not in sys.path:
    sys.path.insert(0, '/content/ML_Project')
os.environ['PYTHONPATH'] = '/content/ML_Project:' + os.environ.get('PYTHONPATH', '')

import torch
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device     : {torch.cuda.get_device_name(0)}")
!nvidia-smi

In [ ]:
# --- Step 2: Verify Checkpoint Paths ---
# Set these paths to where your trained checkpoints are stored in Google Drive or /content

BASELINE_CKPT = "/content/drive/MyDrive/checkpoints/baseline_resnet18/best_baseline_resnet18.pt"
AUGMIX_CKPT   = "/content/drive/MyDrive/checkpoints/augmix_resnet18/best_augmix_resnet18.pt"
FREQ_CKPT     = "/content/drive/MyDrive/checkpoints/frequency_resnet18/best_frequency_resnet18.pt"
CBM_CKPT      = "/content/drive/MyDrive/checkpoints/cbm_resnet18/best_cbm_resnet18.pt"

checkpoints = {
    "Baseline": BASELINE_CKPT,
    "AugMix": AUGMIX_CKPT,
    "Frequency": FREQ_CKPT,
    "CBM": CBM_CKPT
}

for name, path in checkpoints.items():
    exists = os.path.exists(path)
    print(f"[{'FOUND' if exists else 'MISSING'}] {name:10s} : {path}")
    if not exists:
        print(f"   -> Warning: If path is different, please update {name.upper()}_CKPT above.")

In [ ]:
# --- Step 3A: Evaluate Baseline ResNet-18 under Full 75-Condition BN-Adapt ---
!PYTHONPATH=. python scripts/evaluate_full_benchmark_suite.py \
    --checkpoint "$BASELINE_CKPT" \
    --output-json reports/diagnostics/test_time_baseline.json \
    --seeds 42 \
    --corruptions all \
    --mode full_75

In [ ]:
# --- Step 3B: Evaluate AugMix ResNet-18 under Full 75-Condition BN-Adapt ---
!PYTHONPATH=. python scripts/evaluate_full_benchmark_suite.py \
    --checkpoint "$AUGMIX_CKPT" \
    --output-json reports/diagnostics/test_time_augmix.json \
    --seeds 42 \
    --corruptions all \
    --mode full_75

In [ ]:
# --- Step 3C: Evaluate Frequency ResNet-18 under Full 75-Condition BN-Adapt ---
!PYTHONPATH=. python scripts/evaluate_full_benchmark_suite.py \
    --checkpoint "$FREQ_CKPT" \
    --output-json reports/diagnostics/test_time_frequency.json \
    --seeds 42 \
    --corruptions all \
    --mode full_75

In [ ]:
# --- Step 3D: (Optional) Re-evaluate CBM ResNet-18 if needed ---
# If reports/diagnostics/test_time_adaptation.json already exists, this can be skipped.
!PYTHONPATH=. python scripts/evaluate_full_benchmark_suite.py \
    --checkpoint "$CBM_CKPT" \
    --output-json reports/diagnostics/test_time_cbm.json \
    --seeds 42 \
    --corruptions all \
    --mode full_75

In [ ]:
# --- Step 4: Aggregate into Master 2x4 Factorial Matrix & Visualization ---
import json
import pandas as pd
import matplotlib.pyplot as plt

report_files = {
    "Baseline ResNet-18": "reports/diagnostics/test_time_baseline.json",
    "AugMix ResNet-18": "reports/diagnostics/test_time_augmix.json",
    "Frequency ResNet-18": "reports/diagnostics/test_time_frequency.json",
    "Residual CBM ResNet-18": "reports/diagnostics/test_time_cbm.json",
}

rows = []
for model_name, path in report_files.items():
    if not os.path.exists(path):
        # Check if CBM is stored under default filename
        if "CBM" in model_name and os.path.exists("reports/diagnostics/exp4_cbm_full_benchmark.json"):
            path = "reports/diagnostics/exp4_cbm_full_benchmark.json"
        elif "CBM" in model_name and os.path.exists("reports/diagnostics/test_time_adaptation.json"):
            path = "reports/diagnostics/test_time_adaptation.json"
        else:
            continue
    with open(path) as f:
        d = json.load(f)
    sw = d.get("sweep_75", d)
    
    clean_static = sw.get("clean", {}).get("acc_static", 0.0) * 100
    clean_gated  = sw.get("clean", {}).get("acc_gated", clean_static / 100) * 100
    mca_static   = sw.get("mca_static_75", 0.0) * 100
    mca_gated    = sw.get("mca_gated_75", sw.get("mca_adapt_75", 0.0)) * 100
    gain         = mca_gated - mca_static
    
    rows.append({
        "Model Paradigm": model_name,
        "Clean Static (%)": f"{clean_static:.2f}%",
        "Clean Gated (%)": f"{clean_gated:.2f}%",
        "Static mCA (%)": f"{mca_static:.2f}%",
        "Shift-Gated mCA (%)": f"{mca_gated:.2f}%",
        "Net Gain (Δ)": f"{gain:+.2f}%"
    })

df = pd.DataFrame(rows)
print("=" * 85)
print("MASTER 2x4 FACTORIAL GRID: STATIC VS. SHIFT-GATED BN-ADAPTATION")
print("=" * 85)
display(df)

df.to_csv("reports/diagnostics/master_2x4_bn_benchmark.csv", index=False)
!cp reports/diagnostics/master_2x4_bn_benchmark.csv /content/drive/MyDrive/ 2>/dev/null || true
print("Saved master_2x4_bn_benchmark.csv locally and to Google Drive!")